# 03_Silver_Transformations
This notebook reads the Bronze Delta tables (customers, products, orders) created by the Auto Loader streams, applies cleaning, validation, deduplication, joins, and writes the curated Silver tables.
It uses batch processing (`spark.table`) because the Bronze tables are append‑only. In a production pipeline you would use `foreachBatch` for true streaming, but this keeps the example simple and runnable in a notebook.


In [ ]:
from pyspark.sql import functions as F, Window

CATALOG = "ecommerce_catalog"

# ------------------------------------------------------------------
#  Read Bronze tables
# ------------------------------------------------------------------
customers_bronze = spark.table(f"{CATALOG}.bronze.customers")
products_bronze  = spark.table(f"{CATALOG}.bronze.products")
orders_bronze    = spark.table(f"{CATALOG}.bronze.orders")

# ------------------------------------------------------------------
#  Customers cleaning
# ------------------------------------------------------------------
customers_clean = (customers_bronze
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("signup_date", F.to_date(F.col("signup_date")))
    .fillna({"city": "Unknown", "email": "unknown@email.com"})
    .dropDuplicates(["customer_id"]))

# ------------------------------------------------------------------
#  Products cleaning
# ------------------------------------------------------------------
products_clean = (products_bronze
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.upper(F.trim(F.col("category"))))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("stock", F.col("stock").cast("integer"))
    .fillna({"price": 0.0, "stock": 0, "category": "UNKNOWN"})
    .dropDuplicates(["product_id"]))

# ------------------------------------------------------------------
#  Orders cleaning and validation
# ------------------------------------------------------------------
orders_clean = (orders_bronze
    .withColumn("customer_id", F.col("customer_id").cast("integer"))
    .withColumn("product_id", F.col("product_id").cast("integer"))
    .withColumn("quantity", F.col("quantity").cast("integer"))
    .withColumn("order_date", F.to_date(F.col("order_date")))
    .withColumn("status", F.upper(F.trim(F.col("status"))))
    .withColumn("payment_method", F.upper(F.trim(F.col("payment_method"))))
    .fillna({"quantity": 0, "status": "UNKNOWN", "payment_method": "UNKNOWN"})
    .dropDuplicates(["order_id"]))

# ------------------------------------------------------------------
#  Separate valid and rejected orders (quarantine pattern)
# ------------------------------------------------------------------
valid_orders = orders_clean.filter(
    (F.col("quantity") > 0) &
    F.col("customer_id").isNotNull() &
    F.col("product_id").isNotNull() &
    F.col("order_date").isNotNull()
)

rejected_orders = orders_clean.subtract(valid_orders)

# Write rejected orders (optional)
(rejected_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.silver.rejected_orders"))

# ------------------------------------------------------------------
#  Join valid orders with dimension tables
# ------------------------------------------------------------------
silver_orders = (valid_orders
    .join(customers_clean, on="customer_id", how="left")
    .join(products_clean, on="product_id", how="left"))

# Derived columns
silver_orders = (silver_orders
    .withColumn("total_amount", F.col("quantity") * F.col("price"))
    .withColumn("order_year", F.year(F.col("order_date")))
    .withColumn("order_month", F.month(F.col("order_date")))
    .withColumn("order_day", F.dayofmonth(F.col("order_date")))
    .withColumn("order_status_group",
        F.when(F.col("status") == "COMPLETED", "SUCCESS")
         .when(F.col("status") == "CANCELLED", "CANCELLED")
         .otherwise("OTHER")))

# Optional partition handling – reduce to 4 files for downstream efficiency
silver_orders_coalesced = silver_orders.coalesce(4)

# ------------------------------------------------------------------
#  Write Silver tables
# ------------------------------------------------------------------
(silver_orders_coalesced.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.silver.orders"))

(customers_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.silver.customers"))

(products_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.silver.products"))

print("Silver layer tables are ready.")
